# 🐍 Context Engineering with Agent Scratchpad - Microsoft Agent Framework (Python)

## 📋 Scenario Overview

This notebook demonstrates **context engineering** using the Microsoft Agent Framework. We'll build a vacation planning agent that manages conversation context through a persistent scratchpad system, allowing it to remember user preferences and completed tasks even in long conversations.

**Key Concepts:**
- 📝 **Agent Scratchpad**: Persistent memory for tracking preferences and tasks
- 🧠 **Context Management**: Efficiently managing conversation history
- 💾 **Persistent Storage**: File-based memory that survives conversation resets
- 🔄 **Information Retrieval**: Reading from and writing to external storage

## 🏗️ Technical Implementation

### Core Components
- **Agent Framework**: Python implementation of Microsoft's agent orchestration
- **Azure OpenAI**: GPT-4o-mini for natural language understanding
- **Microsoft Entra ID**: Secure keyless authentication
- **File-based Scratchpad**: Markdown file for persistent memory

### Scratchpad Flow
```
User mentions preference → Agent updates scratchpad
                                  ↓
                    vacation_agent_scratchpad.md
                                  ↓
New request → Agent reads scratchpad → Personalized response
```

## ⚙️ Prerequisites

**Required:**
```bash
pip install agent-framework-core azure-identity python-dotenv
```

**Environment Variables (.env):**
```env
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-10-21
```

**Azure RBAC:** "Cognitive Services OpenAI User" role required

Let's build a context-aware agent! 🌟

In [ ]:
# Package check
try:
    import agent_framework
    print("✓ agent-framework-core is installed")
except ImportError:
    print("❌ Please install: pip install agent-framework-core -U")
    raise

In [ ]:
# 📦 Import Required Libraries
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import InteractiveBrowserCredential
from IPython.display import display, HTML, Markdown

In [ ]:
# 🔧 Load Environment Variables
load_dotenv(override=True)

# Verify configuration
print(f"✓ AZURE_OPENAI_ENDPOINT: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"✓ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
print(f"✓ AZURE_OPENAI_API_VERSION: {os.environ.get('AZURE_OPENAI_API_VERSION')}")

# Remove API key if present
if os.environ.get('AZURE_OPENAI_API_KEY'):
    del os.environ['AZURE_OPENAI_API_KEY']
    print("✓ API key removed - using Entra ID authentication")

In [ ]:
# 📝 Agent Scratchpad Implementation
# This provides persistent memory for the agent

class ScratchpadManager:
    """Manages agent scratchpad - persistent memory for user preferences and completed tasks"""
    
    def __init__(self, filepath: str = "vacation_agent_scratchpad.md"):
        self.filepath = Path(filepath)
        # Initialize scratchpad if it doesn't exist
        if not self.filepath.exists():
            self.filepath.write_text("# Agent Scratchpad\n\n## User Preferences\n\n## Completed Tasks\n\n")
    
    def read_scratchpad(self) -> str:
        """Read the current scratchpad contents.
        
        Returns:
            str: The contents of the scratchpad
        """
        return self.filepath.read_text()
    
    def update_scratchpad(self, category: str, content: str) -> str:
        """Update the scratchpad with new information.
        
        Args:
            category: Either 'preferences' or 'tasks'
            content: The content to add
        
        Returns:
            str: Confirmation message
        """
        current_content = self.filepath.read_text()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        if category.lower() == "preferences":
            # Find the preferences section and append
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## User Preferences" in line:
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        elif category.lower() == "tasks":
            # Find the tasks section and append
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## Completed Tasks" in line:
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        
        self.filepath.write_text(current_content)
        return f"✅ Scratchpad updated with {category}: {content}"

# Create the scratchpad manager
scratchpad = ScratchpadManager("vacation_agent_scratchpad.md")

# Create tool functions that the agent can use
def read_scratchpad() -> str:
    """Read the current agent scratchpad to get user's travel preferences and completed tasks."""
    return scratchpad.read_scratchpad()

def update_scratchpad(category: str, content: str) -> str:
    """Update the agent scratchpad with new user's travel preference or completed tasks.
    
    Args:
        category: Category to update - 'preferences' or 'tasks'
        content: The new content to add
    """
    return scratchpad.update_scratchpad(category, content)

print("📝 Scratchpad system initialized")
print(f"   File: {scratchpad.filepath.absolute()}")

In [ ]:
# 🔗 Create Azure OpenAI Chat Client

print("🔐 Authenticating with Azure...")
print("   A browser window will open for you to sign in")

credential = InteractiveBrowserCredential()

openai_chat_client = AzureOpenAIChatClient(
    credential=credential,
    endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
)

print(f"✓ Connected to Azure OpenAI")
print(f"  Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")

In [ ]:
# 🤖 Create the Vacation Planning Agent with Scratchpad Instructions

AGENT_INSTRUCTIONS = """
You are a helpful vacation planning assistant. Your job is to help users plan their perfect vacation.

CRITICAL SCRATCHPAD RULES - YOU MUST FOLLOW THESE:
1. FIRST ACTION: When starting ANY conversation, immediately call read_scratchpad() to check existing preferences
2. AFTER LEARNING PREFERENCES: When user mentions ANY preference (destinations, activities, budget, dates), 
   immediately call update_scratchpad() with category 'preferences'
3. AFTER COMPLETING TASKS: When you finish creating an itinerary or completing any task,
   immediately call update_scratchpad() with category 'tasks'
4. BEFORE NEW ITINERARY: Always call read_scratchpad() before creating any itinerary

EXAMPLES OF WHEN TO UPDATE SCRATCHPAD:
- User says "I love beaches" → update_scratchpad('preferences', 'Loves beach destinations')
- User says "budget is $3000" → update_scratchpad('preferences', 'Budget: $3000 per person for a week')
- You create an itinerary → update_scratchpad('tasks', 'Created Bali itinerary for beach vacation')

PLANNING PROCESS:
1. Read scratchpad first
2. Ask about preferences if not found
3. Update scratchpad with new information
4. Create detailed itineraries
5. Update scratchpad with completed tasks

BE EXPLICIT: Always announce when you're checking or updating the scratchpad.
"""

agent = ChatAgent(
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    tools=[read_scratchpad, update_scratchpad]
)

print("✓ Vacation planning agent created with scratchpad capabilities")

In [ ]:
# 💬 Conversation Part 1: Initial Planning and Preference Gathering

user_inputs_part1 = [
    "I'm thinking about planning a vacation. Can you help me?",
    "I love beach destinations with great food and culture. I enjoy water sports, exploring local markets, and trying authentic cuisine. My budget is around $3000 per person for a week.",
    "That sounds perfect! Please create a detailed itinerary for Bali.",
]

print("🎯 Part 1: Initial planning and preference gathering...\n")

async def run_part1():
    for user_input in user_inputs_part1:
        # Display user message
        html_output = (
            f"<div style='margin:10px 0; padding:12px 15px; border-left:4px solid #4fc3f7; "
            f"background:rgba(128,128,128,0.1); border-radius:4px;'>"
            f"<strong style='color:#4fc3f7;'>👤 User:</strong><br>"
            f"<div style='margin-top:8px;'>{user_input}</div></div>"
        )
        display(HTML(html_output))
        
        # Get agent response
        response = await agent.run(user_input)
        
        # Display agent response
        html_output = (
            f"<div style='margin:10px 0; padding:12px 15px; border-left:4px solid #81c784; "
            f"background:rgba(128,128,128,0.1); border-radius:4px;'>"
            f"<strong style='color:#81c784;'>🤖 VacationPlanner:</strong><br>"
            f"<div style='margin-top:8px; white-space:pre-wrap;'>{response}</div></div>"
        )
        display(HTML(html_output))

await run_part1()

print("\n✅ Part 1 complete. Preferences stored in scratchpad.")

In [ ]:
# 📄 Check Scratchpad Contents After Part 1

print("📝 Current Scratchpad Contents:\n")
scratchpad_contents = scratchpad.read_scratchpad()
display(Markdown(scratchpad_contents))

In [ ]:
# 💬 Conversation Part 2: Destination Change and Context Retrieval

user_inputs_part2 = [
    "Actually, I've changed my mind. I'd prefer to go to the Greek islands instead. Can you create a new itinerary?",
    "What's the weather like there?",
    "What should I pack?",
]

print("🎯 Part 2: Testing context retrieval with destination change...\n")

async def run_part2():
    for user_input in user_inputs_part2:
        # Display user message
        html_output = (
            f"<div style='margin:10px 0; padding:12px 15px; border-left:4px solid #4fc3f7; "
            f"background:rgba(128,128,128,0.1); border-radius:4px;'>"
            f"<strong style='color:#4fc3f7;'>👤 User:</strong><br>"
            f"<div style='margin-top:8px;'>{user_input}</div></div>"
        )
        display(HTML(html_output))
        
        # Get agent response
        response = await agent.run(user_input)
        
        # Display agent response
        html_output = (
            f"<div style='margin:10px 0; padding:12px 15px; border-left:4px solid #81c784; "
            f"background:rgba(128,128,128,0.1); border-radius:4px;'>"
            f"<strong style='color:#81c784;'>🤖 VacationPlanner:</strong><br>"
            f"<div style='margin-top:8px; white-space:pre-wrap;'>{response}</div></div>"
        )
        display(HTML(html_output))

await run_part2()

print("\n✅ Part 2 complete. Agent used scratchpad to maintain context.")

In [ ]:
# 📄 Final Scratchpad Contents

print("📝 Final Scratchpad Contents:\n")
scratchpad_contents = scratchpad.read_scratchpad()
display(Markdown(scratchpad_contents))

## 🎓 Key Takeaways: Context Engineering with Agent Framework

**What We Demonstrated:**
1. ✅ **Persistent Memory**: Preferences survive across conversation turns
2. ✅ **Task Tracking**: Agent maintains record of completed work
3. ✅ **Context Retrieval**: Agent reads scratchpad before generating responses
4. ✅ **File-based Storage**: Simple, inspectable memory system

**How It Works:**
- Agent has access to `read_scratchpad()` and `update_scratchpad()` tools
- Instructions explicitly tell agent when to use these tools
- Markdown file provides human-readable storage
- Timestamps track when information was added

**Comparison with Semantic Kernel:**
- **Agent Framework**: Simple function-based tools
- **Semantic Kernel**: `@kernel_function` decorators with plugins
- **Both**: Support persistent memory patterns
- **Agent Framework**: More straightforward Python implementation

**Real-World Applications:**
- 📝 **Customer Service**: Remember customer history and preferences
- 🏥 **Healthcare**: Track patient information and treatment plans
- 💼 **Project Management**: Maintain project status and decisions
- 🎓 **Education**: Track student progress and learning preferences

**Advantages of Scratchpad Pattern:**
- **Persistence**: Survives conversation resets
- **Transparency**: Human-readable markdown format
- **Simplicity**: No database required
- **Debugging**: Easy to inspect and modify
- **Version Control**: Can be tracked in git

**Limitations:**
- Not suitable for multi-user scenarios (use database instead)
- No built-in search or indexing
- Manual file management required
- No automatic expiration of old data

**Production Enhancements:**
- Replace file storage with database (Cosmos DB, PostgreSQL)
- Add search capabilities with vector embeddings
- Implement automatic summarization of old entries
- Add access control for multi-user systems
- Implement backup and recovery mechanisms

**Next Steps:**
- Explore **Lesson 13** for advanced memory with vector databases
- Combine scratchpad with RAG for semantic search
- Implement memory pruning strategies
- Build multi-agent systems with shared memory

---

**🎉 Congratulations!** You've built a context-aware agent with persistent memory using Microsoft Agent Framework!

In [ ]:
# 🧹 Optional Cleanup
# Uncomment to delete the scratchpad file
# scratchpad.filepath.unlink(missing_ok=True)
# print("🗑️ Scratchpad file deleted")

print("✅ Demo complete! Scratchpad file preserved for review.")